En este notebook se crea el modelo TransR de embeddings con la ontolgía de 3000 animes

In [1]:
import numpy as np
from typing import List, Tuple, Dict
import torch
from pykeen.pipeline import pipeline
from pykeen.triples import TriplesFactory
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
from rdflib import Graph, Namespace, RDF, RDFS, Literal
import sbc_tools as sbc

d:\Users\pedro\Documents\Programacion\Master\RRBC\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def obtener_embedding_entidad(model, entity_name: str, entity_to_id: Dict) -> np.ndarray:
    """
    Obtiene el vector embedding de una entidad.
    Maneja automáticamente si el modelo está en CPU o GPU.
    """
    if entity_name not in entity_to_id:
        raise ValueError(f"Entidad '{entity_name}' no encontrada en el grafo")
    
    entity_id = entity_to_id[entity_name]
    
    # Create tensor on the same device as the model
    device = model.device
    id_tensor = torch.tensor([entity_id], device=device)
    
    # Get embedding
    # TransR stores base entity embeddings in entity_representations[0]
    embedding_tensor = model.entity_representations[0](id_tensor)
    
    # Move to CPU before converting to numpy
    return embedding_tensor.detach().cpu().numpy()[0]

def calcular_similitudes(result, animes: List[str]):
    """
    Calcula la matriz de similitud entre una lista específica de animes.
    """
    model = result.model
    entity_to_id = result.training.entity_to_id
    
    print("\n" + "=" * 70)
    print("CÁLCULO DE SIMILITUDES (TransR)")
    print("=" * 70)
    
    embeddings = []
    animes_validos = []
    
    for anime in animes:
        try:
            emb = obtener_embedding_entidad(model, anime, entity_to_id)
            embeddings.append(emb)
            animes_validos.append(anime)
        except ValueError:
            print(f"⚠ '{anime}' no encontrado (puede tener un nombre diferente en el TTL)")
    
    if not animes_validos:
        return
        
    embeddings = np.array(embeddings)
    
    # Calcular matriz de similitud coseno
    sim_matrix = cosine_similarity(embeddings)
    
    # Mostrar resultados formateados
    print(f"\nMatriz de similitud entre {len(animes_validos)} animes:\n")
    
    # Header
    print(f"{'':25}", end='')
    for p in animes_validos:
        # Truncate long names for display
        print(f"{p[:10]:12}", end='') 
    print()
    
    # Rows
    for i, p1 in enumerate(animes_validos):
        print(f"{p1[:23]:25}", end='')
        for j, p2 in enumerate(animes_validos):
            print(f"{sim_matrix[i][j]:12.4f}", end='')
        print()
    
    return sim_matrix

def encontrar_animes_similares(result, anime_query: str, top_k: int = 5):
    """
    Encuentra los animes más similares usando búsqueda vectorizada (Mucho más rápido).
    """
    model = result.model
    entity_to_id = result.training.entity_to_id
    id_to_entity = {v: k for k, v in entity_to_id.items()}
    
    # 1. Obtener embedding del query
    try:
        query_emb = obtener_embedding_entidad(model, anime_query, entity_to_id)
    except ValueError as e:
        print(f"Error: {e}")
        return

    # 2. Obtener TODOS los embeddings a la vez (en CPU)
    # Esto es mucho más rápido que iterar uno por uno
    all_embeddings = model.entity_representations[0](indices=None).detach().cpu().numpy()
    
    # 3. Calcular similitud coseno contra TODO el grafo
    # Reshape query to (1, dim) for scikit-learn
    sim_scores = cosine_similarity(query_emb.reshape(1, -1), all_embeddings)[0]
    
    # 4. Obtener los índices de los mejores puntajes (orden descendente)
    # Obtenemos top_k + 50 para tener margen de filtrar cosas que no son animes
    best_indices = np.argsort(sim_scores)[::-1][:top_k + 50]
    
    # 5. Filtrar resultados (Excluir clases de ontología y el propio query)
    # Lista de términos de ontología a ignorar
    blacklist = [
        anime_query, 'AnimeSeries', 'Character', 'VoiceActor', 'Studio', 
        'ProductionRole', 'Person', 'RealPerson', 'Group', 'Organization',
        'Director', 'Script', 'Music', 'Producer', 'Creator'
    ]
    
    print(f"\n{'=' * 70}")
    print(f"Top {top_k} animes similares a '{anime_query}':")
    print(f"{'=' * 70}")
    
    count = 0
    for idx in best_indices:
        entity_name = id_to_entity[idx]
        score = sim_scores[idx]
        
        # Filtro simple: Si termina en "_anime" es un género (ej: action__anime), lo saltamos si solo queremos series
        # Si está en la blacklist, lo saltamos
        if entity_name in blacklist or entity_name.endswith("__anime"):
            continue
            
        print(f"{count + 1}. {entity_name:30} (Similitud: {score:.4f})")
        count += 1
        if count >= top_k:
            break

In [9]:
import torch
import numpy as np
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline
from rdflib import Graph
from typing import List, Tuple
import gc # Garbage collection for memory management

def train_optimized_transr(
    ttl_file_path: str,
    embedding_dim: int = 64,     # Reduced from 128 for speed/memory balance
    relation_dim: int = 64,      # TransR specific: projection dimension
    max_epochs: int = 100,       # Cap for epochs
    batch_size: int = 4096,      # Increased for faster throughput
    genre_weight: int = 5        # Reduced slightly to prevent overfitting on genres
):
    # --- STEP 1: Efficient Parsing ---
    print(f"Loading and parsing {ttl_file_path}...")
    g = Graph()
    g = sbc.load(ttl_file_path, format="turtle")
    
    triples_list = []
    
    # Pre-compile string replacements for speed
    namespaces = [
        "http://www.fdi.ucm.es/mia/sbc/animeontology/",
        "http://www.semanticweb.org/eladi/ontologies/anime-ontology#",
        "http://www.w3.org/1999/02/22-rdf-syntax-ns#"
    ]
    
    for s, p, o in g:
        s_str, p_str, o_str = str(s), str(p), str(o)
        
        # Skip schema definitions (Classes, Properties) to focus on data
        if p_str.endswith("type") and o_str in [
            "http://www.w3.org/2002/07/owl#Class", 
            "http://www.w3.org/2002/07/owl#ObjectProperty",
            "http://www.w3.org/2002/07/owl#DatatypeProperty"
        ]:
            continue

        # Fast string cleaning
        for ns in namespaces:
            s_str = s_str.replace(ns, "")
            p_str = p_str.replace(ns, "")
            o_str = o_str.replace(ns, "")
            
        triples_list.append([s_str, p_str, o_str])

    # Free up memory immediately after parsing
    del g
    gc.collect()
    
    print(f"Total Triples Parsed: {len(triples_list)}")

    # --- STEP 2: Factory Creation ---
    tf_all = TriplesFactory.from_labeled_triples(
        np.array(triples_list, dtype=str),
        create_inverse_triples=False # False saves memory, TransR handles direction well
    )
    
    # Split: Train (80%), Validation (10%), Test (10%)
    # Validation is crucial for Early Stopping
    tf_train, tf_test, tf_val = tf_all.split([0.8, 0.1, 0.1], random_state=42)

    # --- STEP 3: Optimized Genre Weighting ---
    # Identify genre relations (usually 'type' or 'hasGenre')
    genre_rel_ids = [
        id for label, id in tf_train.relation_to_id.items() 
        if label in ['type', 'genre', 'hasGenre']
    ]

    if genre_rel_ids:
        print(f"Applying weight x{genre_weight} to genre relations...")
        train_tensor = tf_train.mapped_triples
        
        # Vectorized mask generation (fast)
        # Check if the relation ID (column 1) is in our list of genre IDs
        mask = torch.isin(train_tensor[:, 1], torch.tensor(genre_rel_ids))
        genre_triples = train_tensor[mask]
        
        if len(genre_triples) > 0:
            # Repeat genre triples to oversample them
            repeated = genre_triples.repeat(genre_weight - 1, 1)
            new_triples = torch.cat([train_tensor, repeated], dim=0)
            
            tf_train = TriplesFactory(
                mapped_triples=new_triples,
                entity_to_id=tf_train.entity_to_id,
                relation_to_id=tf_train.relation_to_id,
                create_inverse_triples=False
            )
            print(f"Training set expanded: {len(train_tensor)} -> {len(new_triples)}")

    # --- STEP 4: Optimized Pipeline ---
    # Check for GPU
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Training on: {device}")

    result = pipeline(
        model='TransR',
        training=tf_train,
        testing=tf_test,
        validation=tf_val, # Added validation set
        
        # Model Hyperparameters
        model_kwargs={
            'embedding_dim': embedding_dim,
            'relation_dim': relation_dim, # TransR needs this
            'scoring_fct_norm': 2
        },
        
        # Optimizer
        optimizer='Adam',
        optimizer_kwargs={'lr': 0.001}, # Lower LR for stability with large data
        
        # Negative Sampling (Optimized)
        negative_sampler='bernoulli',
        negative_sampler_kwargs={
            'num_negs_per_pos': 5 # Reduced from 50 to 10 for speed
        },
        
        # Training Loop
        training_kwargs={
            'num_epochs': max_epochs,
            'batch_size': batch_size, # Large batch size for GPU efficiency
            'use_tqdm_batch': False   # Reduces log clutter
        },
        
        # Early Stopping (Crucial for large datasets)
        stopper='early',
        stopper_kwargs={
            'frequency': 5,       # Check every 5 epochs
            'patience': 3,        # Stop if no improvement after 3 checks
            'relative_delta': 0.002
        },
        
        device=device,
        random_seed=42
    )
    
    return result

*Entrenamos el modelo en Google Colab

In [10]:

print("Starting training...")
result = train_optimized_transr(
    "end.ttl", 
    embedding_dim=64, 
    max_epochs=100, 
    batch_size=1024
)
result.save_to_directory("end_model")



Starting training...
Loading and parsing end.ttl...
Total Triples Parsed: 373312


INFO:pykeen.triples.splitting:done splitting triples to groups of sizes [183528, 37330, 37331]
INFO:pykeen.pipeline.api:Using device: cpu
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()


Applying weight x5 to genre relations...
Training set expanded: 298644 -> 582952
Training on: cpu


INFO:pykeen.stoppers.early_stopping:Inferred checkpoint path for best model weights: C:\Users\pedro\.data\pykeen\checkpoints\best-model-weights-7f72b06a-5efd-475a-aa56-8060ec2da65c.pt
d:\Users\pedro\Documents\Programacion\Master\RRBC\venv\Lib\site-packages\torch\utils\data\dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)
Training epochs on cpu:   3%|▎         | 3/100 [06:14<3:22:03, 124.98s/epoch, loss=0.121, prev_loss=0.165]


KeyboardInterrupt: 

In [ ]:
test_list = [
    "One_Piece", 
    "Naruto", 
    "Dragon_Ball", 
    "Cowboy_Bebop", 
    "Neon_Genesis_Evangelion"
]

existing_animes = [a for a in test_list if a in result.training.entity_to_id]
if len(existing_animes) > 1:
    calcular_similitudes(result, existing_animes)

if "One_Piece" in result.training.entity_to_id:
    encontrar_animes_similares(result, "One_Piece", top_k=5)

if "Neon_Genesis_Evangelion" in result.training.entity_to_id:
    encontrar_animes_similares(result, "Neon_Genesis_Evangelion", top_k=5)